In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#cell 1
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠ No GPU — Runtime → Change runtime type → T4 GPU")

print("\n✓ Imports OK")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

✓ Imports OK


In [ ]:
#cell2
DRIVE_DIR  = Path("/content/drive/MyDrive")

DATA_DIR   = DRIVE_DIR / "data_4x"
OUT_DIR    = DRIVE_DIR / "aneo_output"
MODEL_DIR  = DRIVE_DIR / "aneo_models"
TENSOR_DIR = DRIVE_DIR / "aneo_tensors"
FIG_DIR    = DRIVE_DIR / "aneo_figures"

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"split.json exists       : {(TENSOR_DIR/'split.json').exists()}")
print(f"normalization stats     : {(OUT_DIR/'normalization_stats.csv').exists()}")

tensor_files = sorted(TENSOR_DIR.glob("*.npy"))
print(f"Tensor files (.npy)     : {len(tensor_files)}")
print(f"\n✓ Paths OK")

split.json exists       : True
normalization stats     : True
Tensor files (.npy)     : 241

✓ Paths OK


In [ ]:
#cell 3 ── ONE-TIME: create grid_features.npy from aneo_grid_points.csv ─────────────
import pandas as pd
import numpy as np
from pathlib import Path

grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
print("Grid columns :", grid.columns.tolist())
print("Grid shape   :", grid.shape)
print(f"\nz_terrain_m  : {grid['z_terrain_m'].min():.1f} to {grid['z_terrain_m'].max():.1f} m")
print(f"z_rel_m      : {grid['z_rel_m'].min():.1f} to {grid['z_rel_m'].max():.1f} m")
print(f"z0_m         : {grid['z0_m'].min():.4f} to {grid['z0_m'].max():.4f} m")
print(f"slope_deg    : {grid['slope_deg'].min():.1f} to {grid['slope_deg'].max():.1f} °")
print(f"aspect_deg   : {grid['aspect_deg'].min():.1f} to {grid['aspect_deg'].max():.1f} °")

NX, NY = 1000, 500
grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)

def normalize(arr):
    return (arr - arr.mean()) / (arr.std() + 1e-8)

alt_grid     = normalize(grid['z_terrain_m'].values).reshape(NY, NX)
zrel_grid    = normalize(grid['z_rel_m'].values).reshape(NY, NX)
z0_grid      = normalize(grid['z0_m'].values).reshape(NY, NX)
slope_grid   = normalize(grid['slope_deg'].values).reshape(NY, NX)
aspect_rad   = np.deg2rad(grid['aspect_deg'].values)
asp_sin_grid = np.sin(aspect_rad).reshape(NY, NX).astype(np.float32)
asp_cos_grid = np.cos(aspect_rad).reshape(NY, NX).astype(np.float32)
x_grid       = normalize(grid['x_m'].values).reshape(NY, NX)
y_grid       = normalize(grid['y_m'].values).reshape(NY, NX)

# 8 grid features total
grid_features = np.stack([
    alt_grid, zrel_grid, z0_grid, slope_grid,
    asp_sin_grid, asp_cos_grid, x_grid, y_grid,
], axis=-1).astype(np.float32)

np.save(TENSOR_DIR / "grid_features.npy", grid_features)

print(f"\ngrid_features shape : {grid_features.shape}  (NY, NX, 8)")
print(f"✓ grid_features.npy saved ({grid_features.nbytes/1e6:.1f} MB) — run once only")

Grid columns : ['point_id', 'dataset_name', 'grid_name', 'ix', 'iy', 'x_m', 'y_m', 'z_terrain_m', 'z_rel_m', 'z0_m', 'terrain_type', 'land_use_code', 'slope_deg', 'aspect_deg', 'geom_wkt']
Grid shape   : (500000, 15)

z_terrain_m  : 0.0 to 526.5 m
z_rel_m      : 2.0 to 350.0 m
z0_m         : 0.0001 to 2.1912 m
slope_deg    : 0.0 to 24.3 °
aspect_deg   : 0.0 to 360.0 °

grid_features shape : (500, 1000, 8)  (NY, NX, 8)
✓ grid_features.npy saved (16.0 MB) — run once only


In [ ]:
#cell4
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]
NY          = split["ny"]
N_INPUT     = split["n_input"]       # 8 original channels
N_TARGET    = split["n_target"]
INPUT_COLS  = split["input_cols"]
TARGET_COLS = split["target_cols"]

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)

STD_U = norm_stats.loc["wind_u_m_s", "std"]
STD_V = norm_stats.loc["wind_v_m_s", "std"]
STD_W = norm_stats.loc["wind_w_m_s", "std"]
MU_U  = norm_stats.loc["wind_u_m_s", "mean"]
MU_V  = norm_stats.loc["wind_v_m_s", "mean"]
MU_W  = norm_stats.loc["wind_w_m_s", "mean"]

# 8 original + 8 grid + 2 temporal = 18
N_GRID_FEATURES     = 8   # z_terrain, z_rel, z0, slope, asp_sin, asp_cos, x, y
N_TEMPORAL_FEATURES = 2   # hour_sin, hour_cos
N_INPUT_EFFECTIVE   = N_INPUT + N_GRID_FEATURES + N_TEMPORAL_FEATURES  # 18

print("Configuration loaded :\n")
print(f"  Train : {len(TRAIN_DAYS)} days → {TRAIN_DAYS}")
print(f"  Val   : {len(VAL_DAYS)} days → {VAL_DAYS}")
print(f"  Test  : {len(TEST_DAYS)} days → {TEST_DAYS}")
print(f"\n  Grid     : {NX} x {NY}  (dx=dy=50m)")
print(f"  N_INPUT (original)  : {N_INPUT}")
print(f"  N_GRID_FEATURES     : {N_GRID_FEATURES}")
print(f"  N_TEMPORAL_FEATURES : {N_TEMPORAL_FEATURES}")
print(f"  N_INPUT (effective) : {N_INPUT_EFFECTIVE}")
print(f"  N_TARGET : {N_TARGET}")
print(f"\n  STD u,v,w : {STD_U:.4f}, {STD_V:.4f}, {STD_W:.4f}")
print(f"  MU  u,v,w : {MU_U:.4f}, {MU_V:.4f}, {MU_W:.4f}")
print(f"\n✓ Config OK — {N_INPUT_EFFECTIVE} input channels")

Configuration loaded :

  Train : 19 days → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  Val   : 5 days → [20, 21, 22, 24, 25]
  Test  : 6 days → [23, 26, 27, 28, 29, 30]

  Grid     : 1000 x 500  (dx=dy=50m)
  N_INPUT (original)  : 8
  N_GRID_FEATURES     : 8
  N_TEMPORAL_FEATURES : 2
  N_INPUT (effective) : 18
  N_TARGET : 3

  STD u,v,w : 2.0169, 2.1665, 0.0621
  MU  u,v,w : 1.5670, 4.1370, -0.0272

✓ Config OK — 18 input channels


In [ ]:
#cell 5
class ANEODataset(Dataset):
    """
    Input channels: 8 original + 8 grid + 2 temporal = 18 total
    Grid features (static, loaded once):
      z_terrain_m, z_rel_m, z0_m, slope_deg,
      aspect_sin, aspect_cos, x_m, y_m
    Temporal features (per snapshot):
      hour_sin, hour_cos
    Augmentation applied only to original 8 channels.
    """
    def __init__(self, days, heures, tensor_dir, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                if x_path.exists() and y_path.exists():
                    self.samples.append((x_path, y_path, day, h))

        # Load static grid features ONCE — (8, NY, NX)
        gf_path = self.tensor_dir / "grid_features.npy"
        if gf_path.exists():
            gf = np.load(gf_path)  # (NY, NX, 8)
            self.grid_features = torch.from_numpy(
                gf.transpose(2, 0, 1).astype(np.float32))  # (8, NY, NX)
            n_gf = self.grid_features.shape[0]
            print(f"  Dataset : {len(self.samples)} snapshots "
                  f"({'aug ON' if augment else 'aug OFF'})  "
                  f"+{n_gf} grid +2 temporal = {8+n_gf+2} channels")
        else:
            self.grid_features = None
            print(f"  Dataset : {len(self.samples)} snapshots "
                  f"({'aug ON' if augment else 'aug OFF'})  "
                  f"⚠ grid_features.npy not found — run cell 3 first")

    def __len__(self):
        return len(self.samples)

    def augment_sample(self, X, Y):
        X, Y = X.clone(), Y.clone()

        # Wind rotation ±30°
        if torch.rand(1) > 0.5:
            angle = torch.FloatTensor(1).uniform_(-30, 30) * np.pi / 180
            cos_a, sin_a = torch.cos(angle).item(), torch.sin(angle).item()
            u_x, v_x = X[4].clone(), X[5].clone()
            X[4] = cos_a * u_x - sin_a * v_x
            X[5] = sin_a * u_x + cos_a * v_x
            u_y, v_y = Y[0].clone(), Y[1].clone()
            Y[0] = cos_a * u_y - sin_a * v_y
            Y[1] = sin_a * u_y + cos_a * v_y

        # Intensity scaling ±20%
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[4:7] *= scale
            Y      *= scale

        # Gaussian noise on original 8 channels only
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02

        return X, Y

    def __getitem__(self, idx):
        x_path, y_path, day, h = self.samples[idx]

        # Original 8 channels — (8, NY, NX)
        X = torch.from_numpy(
            np.load(x_path).astype(np.float32).transpose(2, 0, 1))
        Y = torch.from_numpy(
            np.load(y_path).astype(np.float32).transpose(2, 0, 1))

        # Augmentation on original 8 channels only
        if self.augment:
            X, Y = self.augment_sample(X, Y)

        # Static grid features — (8, NY, NX)
        if self.grid_features is not None:
            X = torch.cat([X, self.grid_features], dim=0)  # → (16, NY, NX)

        # Temporal encoding — (2, NY, NX)
        _, H, W = X.shape
        hour_sin = float(np.sin(2 * np.pi * h / 24))
        hour_cos = float(np.cos(2 * np.pi * h / 24))
        sin_ch = torch.full((1, H, W), hour_sin, dtype=torch.float32)
        cos_ch = torch.full((1, H, W), hour_cos, dtype=torch.float32)
        X = torch.cat([X, sin_ch, cos_ch], dim=0)  # → (18, NY, NX)

        return X, Y, day, h


print("✓ ANEODataset defined  (8 original + 8 grid + 2 temporal = 18 channels)")

✓ ANEODataset defined  (8 original + 8 grid + 2 temporal = 18 channels)


In [ ]:
#cell 6
BATCH_SIZE = 1

train_dataset = ANEODataset(TRAIN_DAYS + VAL_DAYS, HEURES, TENSOR_DIR, augment=True)
val_dataset   = ANEODataset(VAL_DAYS,   HEURES, TENSOR_DIR, augment=False)
test_dataset  = ANEODataset(TEST_DAYS,  HEURES, TENSOR_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,
                          shuffle=False, num_workers=2)

print(f"\nBatch size     : {BATCH_SIZE}")
print(f"Train batches  : {len(train_loader)}")
print(f"Val   batches  : {len(val_loader)}")
print(f"Test  batches  : {len(test_loader)}")

X_b, Y_b, _, _ = next(iter(train_loader))
print(f"\nX : {X_b.shape}  ← should be [1, {N_INPUT_EFFECTIVE}, 500, 1000]")
print(f"Y : {Y_b.shape}  ← should be [1, 3, 500, 1000]")

assert X_b.shape[1] == N_INPUT_EFFECTIVE, \
    f"X channels={X_b.shape[1]} but expected {N_INPUT_EFFECTIVE}"

print(f"\n✓ Dataloaders ready — {N_INPUT_EFFECTIVE} input channels confirmed")

  Dataset : 96 snapshots (aug ON)  +8 grid +2 temporal = 18 channels
  Dataset : 20 snapshots (aug OFF)  +8 grid +2 temporal = 18 channels
  Dataset : 24 snapshots (aug OFF)  +8 grid +2 temporal = 18 channels

Batch size     : 1
Train batches  : 96
Val   batches  : 20
Test  batches  : 24

X : torch.Size([1, 18, 500, 1000])  ← should be [1, 18, 500, 1000]
Y : torch.Size([1, 3, 500, 1000])  ← should be [1, 3, 500, 1000]

✓ Dataloaders ready — 18 input channels confirmed


In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes_x, modes_y):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes_x      = modes_x
        self.modes_y      = modes_y
        scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(
            scale * torch.rand(in_channels, out_channels,
                               modes_x, modes_y, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        B = x.shape[0]
        x_ft = torch.fft.rfft2(x.float())
        out_ft = torch.zeros(
            B, self.out_channels,
            x.size(-2), x.size(-1) // 2 + 1,
            dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes_x, :self.modes_y] = \
            self.compl_mul2d(
                x_ft[:, :, :self.modes_x, :self.modes_y], self.weights1)
        out_ft[:, :, -self.modes_x:, :self.modes_y] = \
            self.compl_mul2d(
                x_ft[:, :, -self.modes_x:, :self.modes_y], self.weights2)
        return torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))


class FNOBlock(nn.Module):
    def __init__(self, channels, modes_x, modes_y, dropout=0.05):
        super().__init__()
        self.spectral = SpectralConv2d(channels, channels, modes_x, modes_y)
        self.local    = nn.Conv2d(channels, channels, kernel_size=1)
        self.norm     = nn.InstanceNorm2d(channels, affine=True)
        self.dropout  = nn.Dropout2d(p=dropout)

    def forward(self, x):
        return F.gelu(self.dropout(
            self.norm(self.spectral(x) + self.local(x))))


class FNO2D(nn.Module):
    def __init__(self, in_channels, out_channels,
                 hidden_channels=96, n_blocks=5,
                 modes_x=20, modes_y=20, dropout=0.05):
        super().__init__()

        # Lifting
        self.lifting = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1),
            nn.GELU(),
        )

        # FNO blocks
        self.blocks = nn.ModuleList([
            FNOBlock(hidden_channels, modes_x, modes_y, dropout)
            for _ in range(n_blocks)
        ])

        # Global skip connection (negligible memory)
        self.skip_proj = nn.Conv2d(
            hidden_channels, hidden_channels, kernel_size=1)

        # Separate projection heads per component (negligible memory)
        def make_head(h):
            return nn.Sequential(
                nn.Conv2d(h, h//2, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//2, h//4, kernel_size=1), nn.GELU(),
                nn.Conv2d(h//4, 1,   kernel_size=1),
            )

        self.head_u = make_head(hidden_channels)
        self.head_v = make_head(hidden_channels)
        self.head_w = make_head(hidden_channels)

    def forward(self, x):
        x    = self.lifting(x)
        skip = self.skip_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x + skip
        return torch.cat([self.head_u(x),
                          self.head_v(x),
                          self.head_w(x)], dim=1)


model = FNO2D(
    in_channels     = N_INPUT_EFFECTIVE,  # 18
    out_channels    = N_TARGET,           # 3
    hidden_channels = 96,                 # safe for T4
    n_blocks        = 5,
    modes_x         = 20,                 # slight upgrade from 16
    modes_y         = 20,
    dropout         = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FNO2D Model :")
print(f"  in_channels     : {N_INPUT_EFFECTIVE}")
print(f"  hidden_channels : 96")
print(f"  n_blocks        : 5")
print(f"  modes           : 20 x 20  (was 16)")
print(f"  dropout         : 0.05")
print(f"  skip connection : yes")
print(f"  separate heads  : u / v / w")
print(f"  parameters      : {n_params:,}")
print(f"  device          : {device}")
print(f"  on GPU          : {next(model.parameters()).is_cuda}")
print(f"\n✓ Model ready")

FNO2D Model :
  in_channels     : 18
  hidden_channels : 96
  n_blocks        : 5
  modes           : 20 x 20  (was 16)
  dropout         : 0.05
  skip connection : yes
  separate heads  : u / v / w
  parameters      : 36,940,227
  device          : cuda
  on GPU          : True

✓ Model ready


In [ ]:
#cell 8
class PhysicsConstrainedLoss(nn.Module):
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05,
                 lambda_smooth=0.01, lambda_ri=0.02, dx=50.0):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.lambda_ri     = lambda_ri
        self.dx            = dx

    def forward(self, pred, target, rho, phys_weight=1.0):
        L_data = F.mse_loss(pred, target)

        u_pred = pred[:, 0:1, :, :]
        v_pred = pred[:, 1:2, :, :]

        rho_u = rho * u_pred
        rho_v = rho * v_pred
        drhou_dx = (rho_u[:, :, :, 2:] - rho_u[:, :, :, :-2]) / (2 * self.dx)
        drhov_dy = (rho_v[:, :, 2:, :] - rho_v[:, :, :-2, :]) / (2 * self.dx)
        min_y = min(drhou_dx.shape[2], drhov_dy.shape[2])
        min_x = min(drhou_dx.shape[3], drhov_dy.shape[3])
        div = (drhou_dx[:, :, :min_y, :min_x] +
               drhov_dy[:, :, :min_y, :min_x])
        L_mass = (div ** 2).mean()

        w_pred = pred[:, 2:3, :, :]
        L_bc = (w_pred[:, :, 0, :] ** 2).mean()

        grad_u_x = pred[:, 0:1, :, 1:] - pred[:, 0:1, :, :-1]
        grad_u_y = pred[:, 0:1, 1:, :] - pred[:, 0:1, :-1, :]
        grad_v_x = pred[:, 1:2, :, 1:] - pred[:, 1:2, :, :-1]
        grad_v_y = pred[:, 1:2, 1:, :] - pred[:, 1:2, :-1, :]
        L_smooth = ((grad_u_x**2).mean() + (grad_u_y**2).mean() +
                    (grad_v_x**2).mean() + (grad_v_y**2).mean())

        du_dx = (u_pred[:, :, :, 1:] - u_pred[:, :, :, :-1]) / self.dx
        dv_dy = (v_pred[:, :, 1:, :] - v_pred[:, :, :-1, :]) / self.dx
        L_ri = (F.relu(du_dx**2 - 1.0).mean() +
                F.relu(dv_dy**2 - 1.0).mean())

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth +
                     self.lambda_ri     * L_ri)

        return L_data + phys_weight * phys_term, L_data, L_mass, L_bc


criterion = PhysicsConstrainedLoss(
    lambda_mass=0.3, lambda_bc=0.05, lambda_smooth=0.01,
    lambda_ri=0.02, dx=50.0
)
print("✓ Physics loss ready")

✓ Physics loss ready


In [ ]:
# cell 9
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 20
ACCUM_STEPS   = 4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

scaler = GradScaler('cuda')

print("Training config :")
print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}  (effective batch = {BATCH_SIZE*ACCUM_STEPS})")
print(f"  Optimizer     : AdamW  lr=3e-4  wd=1e-4")
print(f"  Scheduler     : CosineAnnealingLR (no restarts), T_max={N_EPOCHS}")
print(f"\n✓ Ready")

Training config :
  N_EPOCHS      : 150
  WARMUP_EPOCHS : 15
  EARLY_STOP    : 20
  ACCUM_STEPS   : 4  (effective batch = 4)
  Optimizer     : AdamW  lr=3e-4  wd=1e-4
  Scheduler     : CosineAnnealingLR (no restarts), T_max=150

✓ Ready


In [ ]:
# cell 10
model.train()
X_b, Y_b, _, _ = next(iter(train_loader))
X_b, Y_b = X_b.to(device), Y_b.to(device)
rho = X_b[:, 1:2, :, :]

with autocast('cuda', dtype=torch.float16):
    pred = model(X_b)
    loss, l_data, l_mass, l_bc = criterion(pred, Y_b, rho, phys_weight=0.0)

scaler.scale(loss).backward()
optimizer.zero_grad()

print(f"L_data : {l_data.item():.6f}")
print(f"GPU memory allocated : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU memory reserved  : {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"\n✓ Single step OK — no OOM")

torch.cuda.empty_cache()

L_data : 0.852696
GPU memory allocated : 0.36 GB
GPU memory reserved  : 9.49 GB

✓ Single step OK — no OOM


In [ ]:
#cell 11
def physics_warmup(epoch, warmup_epochs):
    if warmup_epochs <= 0:
        return 1.0
    return min(1.0, (epoch + 1) / warmup_epochs)


def compute_r2_per_channel(pred, true):
    pred_np = pred.detach().float().cpu().numpy()
    true_np = true.detach().float().cpu().numpy()
    scores = []
    for c in range(pred_np.shape[1]):
        y_true = true_np[:, c].flatten()
        y_pred = pred_np[:, c].flatten()
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        scores.append(1 - ss_res / (ss_tot + 1e-8))
    return np.mean(scores), scores


def train_epoch(epoch):
    model.train()
    train_loss = 0.0
    phys_w = physics_warmup(epoch, WARMUP_EPOCHS)
    optimizer.zero_grad()

    for step, (X_b, Y_b, _, _) in enumerate(train_loader):
        X_b, Y_b = X_b.to(device), Y_b.to(device)
        rho = X_b[:, 1:2, :, :]

        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b)
            loss, l_data, l_mass, l_bc = criterion(
                pred, Y_b, rho, phys_weight=phys_w)
            loss_scaled = loss / ACCUM_STEPS

        loss_scaled.backward()

        if (step+1) % ACCUM_STEPS == 0 or (step+1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item()

    return train_loss / len(train_loader)


def validate_epoch(epoch):
    model.eval()
    val_loss = 0.0
    r2_means = []
    r2_per_ch_sum = np.zeros(3)
    phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

    with torch.no_grad():
        for X_b, Y_b, _, _ in val_loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            rho = X_b[:, 1:2, :, :]

            with autocast('cuda', dtype=torch.float16):
                pred = model(X_b)
                loss, _, _, _ = criterion(pred, Y_b, rho, phys_weight=phys_w)

            val_loss += loss.item()
            r2_mean, r2_ch = compute_r2_per_channel(pred, Y_b)
            r2_means.append(r2_mean)
            r2_per_ch_sum += np.array(r2_ch)

    n = len(val_loader)
    return val_loss / n, np.mean(r2_means), (r2_per_ch_sum / n)


print("✓ physics_warmup / compute_r2 / train_epoch / validate_epoch defined")

✓ physics_warmup / compute_r2 / train_epoch / validate_epoch defined


In [ ]:
#cell 12
history = {'train_loss': [], 'val_loss': [], 'r2': [],
           'r2_u': [], 'r2_v': [], 'r2_w': [],
           'lr': [], 'phys_weight': []}

best_val_loss = float('inf')
best_epoch    = 0
best_r2       = -np.inf
patience_counter = 0

print("="*88)
print("STARTING TRAINING")
print("="*88)
print(f"{'Ep':>4} | {'Train':>9} | {'Val':>9} | {'R²avg':>7} | "
      f"{'R²_u':>6} | {'R²_v':>6} | {'R²_w':>6} | {'φ_w':>5} | {'LR':>8} | Status")
print("-"*88)

try:
    for epoch in range(N_EPOCHS):
        tr_loss = train_epoch(epoch)
        va_loss, r2, r2_ch = validate_epoch(epoch)
        scheduler.step()

        lr     = optimizer.param_groups[0]['lr']
        phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['r2'].append(r2)
        history['r2_u'].append(r2_ch[0])
        history['r2_v'].append(r2_ch[1])
        history['r2_w'].append(r2_ch[2])
        history['lr'].append(lr)
        history['phys_weight'].append(phys_w)

        print(f"{epoch+1:>4} | {tr_loss:>9.5f} | {va_loss:>9.5f} | "
              f"{r2:>7.4f} | {r2_ch[0]:>6.4f} | {r2_ch[1]:>6.4f} | {r2_ch[2]:>6.4f} | "
              f"{phys_w:>5.2f} | {lr:>8.2e} | ", end="")

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_epoch    = epoch + 1
            best_r2       = r2
            patience_counter = 0
            torch.save({
                'epoch':                epoch + 1,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_loss':             va_loss,
                'r2':                   r2,
                'r2_u':                 r2_ch[0],
                'r2_v':                 r2_ch[1],
                'r2_w':                 r2_ch[2],
                'history':              history,
            }, MODEL_DIR / 'best_model_v2.pth')
            print("✓ BEST")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping at epoch {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Training interrupted by user")
except Exception as e:
    print(f"\n✗ Training failed: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "="*88)
print("TRAINING COMPLETE")
print(f"Best epoch : {best_epoch}   "
      f"Best val_loss : {best_val_loss:.6f}   "
      f"Best R² : {best_r2:.4f}")
print("="*88)

pd.DataFrame(history).to_csv(OUT_DIR / 'training_history_v2.csv', index=False)
print(f"\n✓ History saved")

STARTING TRAINING
  Ep |     Train |       Val |   R²avg |   R²_u |   R²_v |   R²_w |   φ_w |       LR | Status
----------------------------------------------------------------------------------------
   1 |   0.85067 |   0.83667 | -0.8530 | -1.5555 | -1.5587 | 0.5552 |  0.07 | 3.00e-04 | ✓ BEST
   2 |   0.73787 |   0.75685 | -0.7784 | -1.4219 | -1.5567 | 0.6433 |  0.13 | 3.00e-04 | ✓ BEST
   3 |   0.67796 |   0.73040 | -0.7817 | -1.3462 | -1.7240 | 0.7250 |  0.20 | 3.00e-04 | ✓ BEST
   4 |   0.65534 |   0.68651 | -0.6893 | -1.2633 | -1.5399 | 0.7354 |  0.27 | 2.99e-04 | ✓ BEST
   5 |   0.61605 |   0.63610 | -0.5761 | -1.1371 | -1.3585 | 0.7673 |  0.33 | 2.99e-04 | ✓ BEST
   6 |   0.52038 |   0.52347 | -0.2957 | -0.7894 | -0.8939 | 0.7964 |  0.40 | 2.99e-04 | ✓ BEST
   7 |   0.40168 |   0.33879 |  0.2249 | -0.1860 | 0.0275 | 0.8332 |  0.47 | 2.98e-04 | ✓ BEST
   8 |   0.24628 |   0.19636 |  0.5814 | 0.3369 | 0.5488 | 0.8585 |  0.53 | 2.98e-04 | ✓ BEST
   9 |   0.15680 |   0.12451 |  0.

In [ ]:
#cell 13
ckpt = torch.load(MODEL_DIR / "best_model_v2.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}  R²_u={ckpt['r2_u']:.4f}  "
      f"R²_v={ckpt['r2_v']:.4f}  R²_w={ckpt['r2_w']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b.to(device))
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U; v_p = pred_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W
        u_t = true_n[0]*STD_U+MU_U; v_t = true_n[1]*STD_V+MU_V
        w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        bias_w = np.mean(w_p-w_t)

        ws_p   = np.sqrt(u_p**2+v_p**2)
        ws_t   = np.sqrt(u_t**2+v_t**2)
        mae_ws = np.mean(np.abs(ws_p-ws_t))

        dir_p   = np.degrees(np.arctan2(v_p, u_p)) % 360
        dir_t   = np.degrees(np.arctan2(v_t, u_t)) % 360
        dir_err = np.abs(dir_p-dir_t)
        dir_err = np.minimum(dir_err, 360-dir_err)
        mae_dir = np.mean(dir_err)

        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v, bias_w=bias_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10,
            acc_w_01=acc_w_01,
        ))

        print(f"  J{day:3d} h{hour:02d} | "
              f"MAE u={mae_u:.4f} v={mae_v:.4f} w={mae_w:.4f} | "
              f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u(±0.5)={acc_u_05:.1f}% Acc_v(±0.5)={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'evaluation_results_v2.csv', index=False)

print(f"\n{'='*72}")
print(f"TEST SET METRICS (mean over {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','bias_u','bias_v','bias_w',
            'mae_ws','mae_dir',
            'acc_u_05','acc_u_10','acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ Results saved: {OUT_DIR/'evaluation_results_v2.csv'}")

Loaded epoch 50  val_loss=0.044776  R²avg=0.9554  R²_u=0.9439  R²_v=0.9530  R²_w=0.9693
  J 23 h00 | MAE u=0.2349 v=0.1928 w=0.0079 | R² u=0.9624 v=0.9646 w=0.9792 | Acc_u(±0.5)=90.1% Acc_v(±0.5)=95.8%
  J 23 h06 | MAE u=0.2153 v=0.1900 w=0.0074 | R² u=0.9571 v=0.9764 w=0.9808 | Acc_u(±0.5)=92.6% Acc_v(±0.5)=96.8%
  J 23 h12 | MAE u=0.2417 v=0.1861 w=0.0087 | R² u=0.9752 v=0.9785 w=0.9818 | Acc_u(±0.5)=88.9% Acc_v(±0.5)=95.9%
  J 23 h18 | MAE u=0.2565 v=0.2031 w=0.0083 | R² u=0.9666 v=0.9361 w=0.9755 | Acc_u(±0.5)=87.8% Acc_v(±0.5)=94.7%
  J 26 h00 | MAE u=0.1596 v=0.1988 w=0.0072 | R² u=0.7275 v=0.9788 w=0.9674 | Acc_u(±0.5)=98.4% Acc_v(±0.5)=95.4%
  J 26 h06 | MAE u=0.1627 v=0.1907 w=0.0071 | R² u=0.5979 v=0.9807 w=0.9603 | Acc_u(±0.5)=98.2% Acc_v(±0.5)=96.2%
  J 26 h12 | MAE u=0.1637 v=0.1887 w=0.0074 | R² u=0.8433 v=0.9872 w=0.9743 | Acc_u(±0.5)=98.1% Acc_v(±0.5)=94.4%
  J 26 h18 | MAE u=0.1746 v=0.1562 w=0.0075 | R² u=0.8724 v=0.9851 w=0.9652 | Acc_u(±0.5)=97.9% Acc_v(±0.5)=97.8%


In [ ]:
#cell 13
ckpt = torch.load(MODEL_DIR / "best_model_v2.pth",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.6f}  "
      f"R²avg={ckpt['r2']:.4f}  R²_u={ckpt['r2_u']:.4f}  "
      f"R²_v={ckpt['r2_v']:.4f}  R²_w={ckpt['r2_w']:.4f}")

results = []
with torch.no_grad():
    for X_b, Y_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        with autocast('cuda', dtype=torch.float16):
            pred = model(X_b.to(device))
        pred_n = pred[0].float().cpu().numpy()
        true_n = Y_b[0].numpy()

        u_p = pred_n[0]*STD_U+MU_U; v_p = pred_n[1]*STD_V+MU_V
        w_p = pred_n[2]*STD_W+MU_W
        u_t = true_n[0]*STD_U+MU_U; v_t = true_n[1]*STD_V+MU_V
        w_t = true_n[2]*STD_W+MU_W

        mae_u  = np.mean(np.abs(u_p-u_t))
        mae_v  = np.mean(np.abs(v_p-v_t))
        mae_w  = np.mean(np.abs(w_p-w_t))
        rmse_u = np.sqrt(np.mean((u_p-u_t)**2))
        rmse_v = np.sqrt(np.mean((v_p-v_t)**2))
        rmse_w = np.sqrt(np.mean((w_p-w_t)**2))
        r2_u   = 1-np.sum((u_p-u_t)**2)/(np.sum((u_t-u_t.mean())**2)+1e-9)
        r2_v   = 1-np.sum((v_p-v_t)**2)/(np.sum((v_t-v_t.mean())**2)+1e-9)
        r2_w   = 1-np.sum((w_p-w_t)**2)/(np.sum((w_t-w_t.mean())**2)+1e-9)
        bias_u = np.mean(u_p-u_t)
        bias_v = np.mean(v_p-v_t)
        bias_w = np.mean(w_p-w_t)

        ws_p   = np.sqrt(u_p**2+v_p**2)
        ws_t   = np.sqrt(u_t**2+v_t**2)
        mae_ws = np.mean(np.abs(ws_p-ws_t))

        dir_p   = np.degrees(np.arctan2(v_p, u_p)) % 360
        dir_t   = np.degrees(np.arctan2(v_t, u_t)) % 360
        dir_err = np.abs(dir_p-dir_t)
        dir_err = np.minimum(dir_err, 360-dir_err)
        mae_dir = np.mean(dir_err)

        acc_u_05 = (np.abs(u_p-u_t)<0.5).mean()*100
        acc_u_10 = (np.abs(u_p-u_t)<1.0).mean()*100
        acc_v_05 = (np.abs(v_p-v_t)<0.5).mean()*100
        acc_v_10 = (np.abs(v_p-v_t)<1.0).mean()*100
        acc_w_01 = (np.abs(w_p-w_t)<0.1).mean()*100

        results.append(dict(
            day=day, hour=hour,
            mae_u=mae_u, mae_v=mae_v, mae_w=mae_w,
            rmse_u=rmse_u, rmse_v=rmse_v, rmse_w=rmse_w,
            r2_u=r2_u, r2_v=r2_v, r2_w=r2_w,
            bias_u=bias_u, bias_v=bias_v, bias_w=bias_w,
            mae_ws=mae_ws, mae_dir=mae_dir,
            acc_u_05=acc_u_05, acc_u_10=acc_u_10,
            acc_v_05=acc_v_05, acc_v_10=acc_v_10,
            acc_w_01=acc_w_01,
        ))

        print(f"  J{day:3d} h{hour:02d} | "
              f"MAE u={mae_u:.4f} v={mae_v:.4f} w={mae_w:.4f} | "
              f"R² u={r2_u:.4f} v={r2_v:.4f} w={r2_w:.4f} | "
              f"Acc_u(±0.5)={acc_u_05:.1f}% Acc_v(±0.5)={acc_v_05:.1f}%")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'evaluation_results_v2.csv', index=False)

print(f"\n{'='*72}")
print(f"TEST SET METRICS (mean over {len(df_r)} snapshots):")
print(f"{'='*72}")
for col in ['mae_u','mae_v','mae_w','rmse_u','rmse_v','rmse_w',
            'r2_u','r2_v','r2_w','bias_u','bias_v','bias_w',
            'mae_ws','mae_dir',
            'acc_u_05','acc_u_10','acc_v_05','acc_v_10','acc_w_01']:
    print(f"  {col:12s}: {df_r[col].mean():.4f}")
print('='*72)
print(f"\n✓ Results saved: {OUT_DIR/'evaluation_results_v2.csv'}")

Loaded epoch 50  val_loss=0.044776  R²avg=0.9554  R²_u=0.9439  R²_v=0.9530  R²_w=0.9693
  J 23 h00 | MAE u=0.2349 v=0.1928 w=0.0079 | R² u=0.9624 v=0.9646 w=0.9792 | Acc_u(±0.5)=90.1% Acc_v(±0.5)=95.8%
  J 23 h06 | MAE u=0.2153 v=0.1900 w=0.0074 | R² u=0.9571 v=0.9764 w=0.9808 | Acc_u(±0.5)=92.6% Acc_v(±0.5)=96.8%
  J 23 h12 | MAE u=0.2417 v=0.1861 w=0.0087 | R² u=0.9752 v=0.9785 w=0.9818 | Acc_u(±0.5)=88.9% Acc_v(±0.5)=95.9%
  J 23 h18 | MAE u=0.2565 v=0.2031 w=0.0083 | R² u=0.9666 v=0.9361 w=0.9755 | Acc_u(±0.5)=87.8% Acc_v(±0.5)=94.7%
  J 26 h00 | MAE u=0.1596 v=0.1988 w=0.0072 | R² u=0.7275 v=0.9788 w=0.9674 | Acc_u(±0.5)=98.4% Acc_v(±0.5)=95.4%
  J 26 h06 | MAE u=0.1627 v=0.1907 w=0.0071 | R² u=0.5979 v=0.9807 w=0.9603 | Acc_u(±0.5)=98.2% Acc_v(±0.5)=96.2%
  J 26 h12 | MAE u=0.1637 v=0.1887 w=0.0074 | R² u=0.8433 v=0.9872 w=0.9743 | Acc_u(±0.5)=98.1% Acc_v(±0.5)=94.4%
  J 26 h18 | MAE u=0.1746 v=0.1562 w=0.0075 | R² u=0.8724 v=0.9851 w=0.9652 | Acc_u(±0.5)=97.9% Acc_v(±0.5)=97.8%
